In [1]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")
len(answers)

565

In [2]:
from typing import Literal

from pydantic import BaseModel, Field


class AnswerEvaluation(BaseModel):
    reasoning: str = Field(description="Reasoning about the quality of the answer.")
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [5]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question, answer_orig=answer_orig, answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client, aqa_judge_instructions, prompt, AnswerEvaluation, model=model
    )

    return result, usage

In [6]:
rec = answers[0]
eval_result, usage = evaluate_aqa(
    question=rec["question"], answer_orig=rec["answer_orig"], answer_llm=rec["answer_llm"]
)
eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says late joining is allowed and correctly notes that certificate eligibility depends on submitting the project before submissions close. It preserves the key condition and adds no contradictory information.', score='good')

In [7]:
calc_price(usage)

{'input_cost': 0.000231,
 'output_cost': 0.000252,
 'total_cost': 0.00048300000000000003}

In [8]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"], answer_orig=rec["answer_orig"], answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [9]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/565 [00:00<?, ?it/s]

In [10]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

df_eval = pd.DataFrame(evaluations)
calc_total_price(usages)

0.3993030000000001

In [11]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 538/565 = 95.22%


In [12]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
2,What do I need to do to qualify for a certific...,74eb249bbf,bad,The AI answer adds requirements not present in...
18,"Do I need to use the videos, notebooks, and Gi...",04919992b3,bad,The AI answer correctly says you can mix the m...
32,What do I actually need to pass in order to ge...,9f689c185f,bad,The AI answer correctly states that the Capsto...
33,"Is homework required for certification, or is ...",9f689c185f,bad,The AI answer contradicts the ground truth. Th...
39,Why do the homework instructions or files some...,96286b4be4,bad,The AI answer explains that homework materials...


In [13]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)